# Week 2 — Dense Retrieval
**Model:** `all-MiniLM-L6-v2`  
**Dataset:** FiQA Test Set (648 queries, 57,600 passages)  
**Metrics:** NDCG@10, MRR, Recall@10  
**Saves:** FAISS index + results CSVs (loaded by Hybrid_RRF.ipynb)

## Cell 0 — Repo Root & Config

In [ ]:
import sys
import os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / "config.py").exists():
        break
    _root = _root.parent

os.chdir(_root)
sys.path.insert(0, str(_root))

from config import FIQA_CORPUS, FIQA_QUERIES, FIQA_QRELS_TEST

DENSE_RESULTS_DIR = os.path.join("dense_retrieval", "Results")
INDEXES_DIR       = os.path.join(DENSE_RESULTS_DIR, "indexes")
os.makedirs(INDEXES_DIR, exist_ok=True)

print(f"Repo root       : {_root}")
print(f"FIQA_CORPUS     : {FIQA_CORPUS}")
print(f"FIQA_QUERIES    : {FIQA_QUERIES}")
print(f"FIQA_QRELS_TEST : {FIQA_QRELS_TEST}")
print(f"Results dir     : {DENSE_RESULTS_DIR}")
print(f"Indexes dir     : {INDEXES_DIR}")

## Cell 1 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q sentence-transformers faiss-cpu pytrec-eval-terrier

## Cell 2 — Imports

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import faiss
import pytrec_eval
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

print("All imports OK")
print(f"faiss version   : {faiss.__version__}")

## Cell 3 — Load FiQA Corpus, Queries & Qrels

In [ ]:
# ── Corpus ──────────────────────────────────────────────────────────────────
corpus_df = pd.read_csv(FIQA_CORPUS, usecols=["_id", "text"])
corpus_df = corpus_df.dropna(subset=["text"]).reset_index(drop=True)
corpus_df["_id"] = corpus_df["_id"].astype(str)
corpus_texts = corpus_df["text"].tolist()
corpus_ids   = corpus_df["_id"].tolist()

print(f"Corpus loaded   : {len(corpus_df):,} passages")

# ── Queries ──────────────────────────────────────────────────────────────────
queries_df = pd.read_csv(FIQA_QUERIES, usecols=["_id", "text"])
queries_df["_id"] = queries_df["_id"].astype(str)

# ── Qrels ────────────────────────────────────────────────────────────────────
qrels_df = pd.read_csv(FIQA_QRELS_TEST)
qrels_df.columns = [c.strip() for c in qrels_df.columns]
qrels_df = qrels_df.astype(str)

query_col  = [c for c in qrels_df.columns if "query"  in c.lower()][0]
corpus_col = [c for c in qrels_df.columns if "corpus" in c.lower()][0]
score_col  = [c for c in qrels_df.columns if "score"  in c.lower()][0]

qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row[query_col])
    did = str(row[corpus_col])
    qrels_dict.setdefault(qid, {})[did] = int(float(row[score_col]))

test_query_ids  = set(qrels_dict.keys())
test_queries_df = queries_df[queries_df["_id"].isin(test_query_ids)].reset_index(drop=True)

print(f"Test queries    : {len(test_queries_df):,}")
print(f"Qrels dict      : {len(qrels_dict):,} queries")

## Cell 4 — Helper Functions

In [ ]:
def build_faiss_index(model, texts, batch_size=256):
    """Encode texts and build FAISS IndexFlatIP (cosine similarity via L2-norm)."""
    print(f"  Encoding {len(texts):,} passages  (batch_size={batch_size}) ...")
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))
    print(f"  FAISS index built  dim={dim}  vectors={index.ntotal:,}")
    return index


def dense_retrieve(model, index, query_texts, query_ids, top_k=10, batch_size=128):
    """Encode queries, search FAISS index, return {qid: {doc_id: score}}."""
    q_embs = model.encode(
        query_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    scores_matrix, idx_matrix = index.search(q_embs, top_k)

    results = {}
    for i, qid in enumerate(query_ids):
        results[qid] = {
            corpus_ids[idx]: float(scores_matrix[i, rank])
            for rank, idx in enumerate(idx_matrix[i])
            if idx != -1
        }
    return results


def evaluate(results_dict, qrels_dict):
    """Run pytrec_eval, return (aggregated_metrics, per_query_metrics)."""
    evaluator   = pytrec_eval.RelevanceEvaluator(
        qrels_dict, {"ndcg_cut_10", "recip_rank", "recall_10"}
    )
    eval_res    = evaluator.evaluate(results_dict)
    aggregated  = {
        "NDCG@10"   : round(np.mean([v["ndcg_cut_10"] for v in eval_res.values()]), 4),
        "MRR"       : round(np.mean([v["recip_rank"]  for v in eval_res.values()]), 4),
        "Recall@10" : round(np.mean([v["recall_10"]   for v in eval_res.values()]), 4),
    }
    return aggregated, eval_res


def save_results(results_dict, eval_res, test_queries_df, prefix, results_dir):
    """Save top-K results CSV and per-query CSV — same format as baseline."""
    rows = [
        {"query_id": qid, "rank": rank + 1, "doc_id": doc_id, "score": round(score, 6)}
        for qid, docs in results_dict.items()
        for rank, (doc_id, score) in enumerate(
            sorted(docs.items(), key=lambda x: x[1], reverse=True)
        )
    ]
    pd.DataFrame(rows).to_csv(os.path.join(results_dir, f"{prefix}_results.csv"), index=False)

    pq_rows = []
    for qid, scores in eval_res.items():
        qt = test_queries_df[test_queries_df["_id"] == qid]["text"].values
        pq_rows.append({
            "query_id"  : qid,
            "query"     : qt[0] if len(qt) else "",
            "NDCG@10"   : round(scores["ndcg_cut_10"], 4),
            "MRR"       : round(scores["recip_rank"],  4),
            "Recall@10" : round(scores["recall_10"],   4),
        })
    pd.DataFrame(pq_rows).sort_values("NDCG@10", ascending=False).to_csv(
        os.path.join(results_dir, f"{prefix}_per_query.csv"), index=False
    )
    print(f"  Saved {prefix}_results.csv and {prefix}_per_query.csv")


print("Helper functions defined.")

## Cell 5 — Load Model & Build FAISS Index
`all-MiniLM-L6-v2` — 384-dim sentence embedding model trained on MS MARCO + NLI.  
Fast on CPU, strong retrieval quality.

In [ ]:
print("Loading all-MiniLM-L6-v2 ...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"  Embedding dim : {model.get_sentence_embedding_dimension()}")

print("\nBuilding FAISS index ...")
index = build_faiss_index(model, corpus_texts, batch_size=256)

# Save index + corpus_ids so Hybrid_RRF.ipynb can reload without re-encoding
faiss.write_index(index, os.path.join(INDEXES_DIR, "faiss_minilm.index"))
with open(os.path.join(INDEXES_DIR, "corpus_ids.pkl"), "wb") as f:
    pickle.dump(corpus_ids, f)

print("  faiss_minilm.index saved")
print("  corpus_ids.pkl saved")

## Cell 6 — Retrieve & Evaluate

In [ ]:
TOP_K = 10

print(f"Running dense retrieval  (top-{TOP_K}) ...")
results = dense_retrieve(
    model, index,
    test_queries_df["text"].tolist(),
    test_queries_df["_id"].tolist(),
    top_k=TOP_K
)

agg, per_query = evaluate(results, qrels_dict)

print("\n" + "="*45)
print("      MiniLM DENSE RETRIEVAL RESULTS")
print("="*45)
print(f"  Queries evaluated : {len(per_query)}")
print(f"  NDCG@10           : {agg['NDCG@10']}")
print(f"  MRR               : {agg['MRR']}")
print(f"  Recall@10         : {agg['Recall@10']}")
print("="*45)

save_results(results, per_query, test_queries_df, "minilm", DENSE_RESULTS_DIR)

## Cell 7 — Comparison Table & Save dense_metrics.json

In [ ]:
# Load BM25 baseline
with open(os.path.join("baseline", "Results", "baseline_metrics.json")) as f:
    bm25 = json.load(f)

comparison = {
    "dataset"     : "FiQA Test",
    "num_queries" : len(per_query),
    "models": {
        "BM25 Baseline": {"NDCG@10": bm25["NDCG@10"], "MRR": bm25["MRR"], "Recall@10": bm25["Recall@10"]},
        "MiniLM-Dense" : agg,
    }
}

with open(os.path.join(DENSE_RESULTS_DIR, "dense_metrics.json"), "w") as f:
    json.dump(comparison, f, indent=2)
print(f"Saved: dense_retrieval/Results/dense_metrics.json")

print("\n" + "="*50)
print("      WEEK 2 — DENSE RETRIEVAL COMPARISON")
print("="*50)
print(f"  {'Model':<20} {'NDCG@10':>8} {'MRR':>8} {'Recall@10':>10}")
print("-"*50)
for name, m in comparison["models"].items():
    print(f"  {name:<20} {m['NDCG@10']:>8.4f} {m['MRR']:>8.4f} {m['Recall@10']:>10.4f}")
print("="*50)
print("\nNext → Hybrid_RRF.ipynb")

## Cell 8 — Saved Files Summary

In [ ]:
print("Files saved:")
for dirpath, _, filenames in os.walk(DENSE_RESULTS_DIR):
    for fname in sorted(filenames):
        fpath = os.path.join(dirpath, fname)
        size  = os.path.getsize(fpath)
        rel   = os.path.relpath(fpath, _root)
        print(f"  {rel:<55}  {size/1024:>7.1f} KB")